# Neo4j Simple Connection Test

Validates connectivity and creates the UC JDBC connection.

**Prerequisites:**
- Run `sample-validation/upload_data.sh` to upload CSV files to the UC Volume
- Run `sample-validation/create_secrets.sh` to store Neo4j credentials as Databricks secrets
- Run `00-load-graph.ipynb` to load the aircraft graph into Neo4j

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Edit these values for your environment
# =============================================================================

# --- Neo4j Aura ---
NEO4J_URI = "neo4j+s://<instance>.databases.neo4j.io"
SECRET_SCOPE = "sample_validation"
NEO4J_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_USERNAME")
NEO4J_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_PASSWORD")

# --- Databricks Unity Catalog ---
UC_CATALOG = "<catalog>"
UC_SCHEMA = "neo4j_getting_started"
UC_VOLUME = "aircraft_data"
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_HOST = NEO4J_URI.replace("neo4j+s://", "")
NEO4J_JDBC_URL_SQL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/neo4j?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

---

## Section 1: Load sensor_readings Delta Table

Creates the `sensor_readings` table from the CSV file in the UC Volume.
This table is used by the federated queries in notebooks 02 and 03.

**Schema:** `reading_id, sensor_id, ts, value` — 172,800 hourly readings from 160 sensors over 45 days.

In [ ]:
print("--- Section 1: Load sensor_readings Delta Table ---")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQN}.`{UC_VOLUME}`")

spark.sql(f"""
    CREATE OR REPLACE TABLE {FQN}.sensor_readings AS
    SELECT * EXCEPT (_rescued_data)
    FROM read_files('{VOLUME_PATH}/nodes_readings.csv',
        format => 'csv',
        header => true,
        inferColumnTypes => true)
""")

count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.sensor_readings").collect()[0]["cnt"]
status = "PASS" if count == 172800 else "FAIL"
print(f"  [{status}] sensor_readings: {count:,} rows")

spark.sql(f"SELECT sensor_id, COUNT(*) AS readings FROM {FQN}.sensor_readings GROUP BY sensor_id LIMIT 5").show(truncate=False)

---

## Section 2: Create UC JDBC Connection

Creates the Unity Catalog JDBC connection for SQL-to-Cypher federation.
The SafeSpark sandbox wraps the JDBC driver in an isolated JVM.

**Note:** `java_dependencies` only accepts UC Volume paths.

In [ ]:
import time

print("--- Section 2: Create UC JDBC Connection ---")
print(f"  Connection: {UC_CONNECTION_NAME}")
print(f"  URL:        {NEO4J_JDBC_URL_SQL}")

spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")

esc = lambda s: s.replace("'", "\\'")

create_sql = f"""
    CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
    ENVIRONMENT (
        java_dependencies '{JAVA_DEPENDENCIES}'
    )
    OPTIONS (
        url '{esc(NEO4J_JDBC_URL_SQL)}',
        user '{esc(NEO4J_USERNAME)}',
        password '{esc(NEO4J_PASSWORD)}',
        driver 'org.neo4j.jdbc.Neo4jDriver',
        externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
    )
"""

start = time.time()
spark.sql(create_sql)
elapsed = (time.time() - start) * 1000

df = (
    spark.read.format("jdbc")
    .option("databricks.connection", UC_CONNECTION_NAME)
    .option("query", "SELECT 1 AS test")
    .option("customSchema", "test INT")
    .load()
)
test_val = df.collect()[0]["test"]
status = "PASS" if test_val == 1 else "FAIL"
print(f"  [PASS] Connection created in {elapsed:.0f}ms")
print(f"  [{status}] SELECT 1 returned {test_val}")

---

## Section 3: Query via UC JDBC

Runs SQL queries against the Neo4j graph through the Unity Catalog JDBC connection.
SQL is automatically translated to Cypher by the connector.

**Note:** `customSchema` is required because Neo4j's JDBC driver returns `NullType` during Spark's schema inference.

In [ ]:
print("--- Section 3: Query via UC JDBC ---")

def read_neo4j(custom_schema, query):
    return (
        spark.read.format("jdbc")
        .option("databricks.connection", UC_CONNECTION_NAME)
        .option("customSchema", custom_schema)
        .option("query", query)
        .load()
    )

# Aircraft count
df = read_neo4j("aircraft_count LONG", "SELECT COUNT(*) AS aircraft_count FROM Aircraft")
count = df.collect()[0]["aircraft_count"]
status = "PASS" if count == 20 else "FAIL"
print(f"  [{status}] Aircraft: {count}")

# Airport count
df = read_neo4j("airport_count LONG", "SELECT COUNT(*) AS airport_count FROM Airport")
count = df.collect()[0]["airport_count"]
status = "PASS" if count == 12 else "FAIL"
print(f"  [{status}] Airports: {count}")

# Flights by operator
print("\n  Flights by operator:")
read_neo4j(
    "operator STRING, flight_count LONG",
    "SELECT operator, COUNT(*) AS flight_count FROM Flight GROUP BY operator ORDER BY flight_count DESC",
).show(truncate=False)

print("Status: PASS — run 02-federated-queries.ipynb next")